# `=== Atelier Scikit-learn ===`

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Importation du dataset
df = pd.read_csv("../data/mesures_capteurs.csv")

# Exploration du dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    str    
 1   date_heure    605 non-null    str    
 2   id_capteur    605 non-null    str    
 3   batiment      605 non-null    str    
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    str    
dtypes: float64(4), str(5)
memory usage: 42.7 KB


## `Partie 1 – Gestion des doublons`

### `1.1: Vérification de l’existence de doublons dans df`

In [10]:
print(f" Duplications: {df.duplicated().sum()}")

 Duplications: 5


### `1.2: Suppression des doublons`

In [13]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

### `1.3: gestion des Valeurs manquantes`

In [14]:
df.isna().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     6
humidite        5
pression        5
consommation    5
etat            4
dtype: int64

In [15]:
# Statistiques descriptives
df.describe()

,temperature,humidite,pression,consommation
count,594.000000,595.000000,595.000000,595.000000
mean,24.896212,64.863395,1012.241059,208.693294
std,4.068454,10.760478,10.623413,72.320179
min,-18.500000,28.520000,850.000000,18.120000
25%,22.597500,58.090000,1006.890000,160.500000
50%,24.870000,65.370000,1012.960000,206.130000
75%,27.295000,71.590000,1017.835000,254.265000
max,58.700000,145.000000,1038.430000,875.000000


In [16]:
# Gestion des veleurs manquantes pour la variable numeriques
# colonne température
df['temperature'] = df['temperature'].fillna(df['temperature'].median())
# Colonne consommation
df['consommation'] = df['consommation'].fillna(df['consommation'].median())
# Colonne humidite
df['humidite'] = df['humidite'].fillna(df['humidite'].median())
# Colonne pression
df['pression'] = df['pression'].fillna(df['pression'].median())
df.info()

<class 'pandas.DataFrame'>
Index: 600 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     600 non-null    str    
 1   date_heure    600 non-null    str    
 2   id_capteur    600 non-null    str    
 3   batiment      600 non-null    str    
 4   temperature   600 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          596 non-null    str    
dtypes: float64(4), str(5)
memory usage: 46.9 KB


In [17]:
# Gestion des veleurs manquantes pour la variable numeriques
df["etat"].mode()

0    OK
Name: etat, dtype: str

In [18]:
df["etat"] = df["etat"].fillna("OK")
df.isnull().sum()

id_mesure       0
date_heure      0
id_capteur      0
batiment        0
temperature     0
humidite        0
pression        0
consommation    0
etat            0
dtype: int64

## `Partie 2 – Sélection de y (cible) et X (caractéristiques)`

### `2.1: On définit "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives`

`Selection des features`

In [25]:
X = df[['temperature', 'humidite', 'pression', 'consommation']]

`Selection des labels`

In [26]:
y = df[['etat']]


### `2.2: Affichage des cinq premières lignes de X et de y`

In [27]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [28]:
y.head()

,etat
0,OK
1,OK
2,OK
3,OK
4,OK


### `2.3: Quel est le type du problème de machine learning ?`

Le type de problème en question est un problème de classification, l'objectif est de classer les enreigistrements dans les classe: `OK`, `ALERTE` et `ERREUR` en fontion des valeurs de la temperature, de l'humidité, de la pression et de la consommation

## `Partie 3 – Découpage Train/Test`

#### `3.1:` Division de `X `en deux ensembles distincts : un pour l'entraînement `(train)` et un pour le test `(test)`. Avec les conditions suivantes : `20%` des données serviront au test ; garantir la `reproductibilité` du découpage ; conserver les mêmes `proportions de classes` dans l'ensemble de `train` et de `test` que dans les données d'origine`

In [33]:
from sklearn.model_selection import train_test_split

# Division en train et test des features et labels
X_train, X_test, y_train, y_test = train_test_split(
     X,                  # Features
     y,                  # Target
     test_size=0.2,      # 20% des données pour le test
     random_state=42,    # Garantir la reproductibilité du découpage
     stratify=y          # conserver les meme proportions de classe dans train et test
)

## `Partie 4 – Gestion des valeurs manquantes`

### `4.1: Vérification de l’existence de valeurs manquantes`

In [29]:
X.isnull().sum()

temperature     0
humidite        0
pression        0
consommation    0
dtype: int64

### `4.2: Sélection de SimpleImputer avec la médiane`

In [34]:
from sklearn.impute import SimpleImputer

# Instanciation de l'imputeur
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

### `4.3: Qu’est ce qui justifie le choix de la médiane ?`

Le dataset en question contient des outliers (valeurs abérantes) et cela influx sur le calcul de la moyenne et de la variance, et pas sur la mediane; ce qui justifie le choix de la médiane

### `4.4: Les paramètres (médianes) de l’imputeur sur X_train`

In [39]:
imputer

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](4,)","['temperature','humidite','pression','consommation']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](4,)","[ 24.87, 65.33,1012.67, 206.13]"


### `4.5: On déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test`

In [37]:
X_train_imputed

array([[  58.7 ,   81.51, 1026.79,  271.92],
       [  26.2 ,   76.27, 1016.74,  191.33],
       [  27.27,   75.68, 1013.78,  335.2 ],
       ...,
       [  21.64,   52.74, 1002.78,  157.77],
       [  26.27,   68.79, 1003.12,  244.9 ],
       [  32.72,   53.28, 1010.05,  195.37]], shape=(480, 4))

In [38]:
X_test_imputed

array([[  19.27,   58.39, 1003.74,  136.56],
       [  25.96,   62.79, 1012.17,  228.74],
       [  24.87,   69.97, 1014.14,  181.07],
       [  33.29,   50.43, 1009.87,  356.76],
       [  30.38,   65.98, 1005.48,  256.18],
       [  27.85,   58.01, 1012.29,  208.09],
       [  26.04,   67.17, 1011.38,  253.99],
       [  28.66,   35.83, 1022.52,  142.86],
       [  25.02,   79.27, 1027.7 ,  214.88],
       [  20.71,   72.01, 1002.15,  144.24],
       [  24.86,   68.77, 1003.98,  289.79],
       [  25.91,   49.2 , 1016.57,  201.37],
       [  29.5 ,   81.47, 1015.48,  256.53],
       [  27.33,   89.45, 1001.98,  346.24],
       [  22.96,   42.4 , 1011.47,  142.67],
       [  23.64,   48.79,  994.25,  222.42],
       [  20.72,   75.27,  997.63,  230.8 ],
       [  22.79,   68.34, 1013.41,  115.55],
       [  30.92,   67.19, 1009.71,  329.78],
       [  26.06,   77.13, 1021.87,  108.36],
       [  29.97,   68.65, 1007.19,  267.37],
       [  24.3 ,   58.75, 1023.6 ,  244.86],
       [  